Automated Global Database of Reproduction Numbers
Pipeline:

PubMed abstracts
     ↓
LLM extraction
     ↓
Verification agent
     ↓
Structured dataset
     ↓
Meta-analysis + dashboard



How consistent are R₀ estimates across studies?



Title: Parallel Multi-Agent Extraction Pipeline (Dual GPU)

This notebook implements a high-throughput pipeline using two GPUs simultaneously.

Architecture: Producer-Consumer (Pipeline).

GPU 0 (Producer): Runs Agent A (Qwen 2.5-32B). It reads raw text and "produces" extractions.

GPU 1 (Consumer): Runs Agent B (Gemma 2-27B). It consumes extractions and audits them.

Advantage: No model loading/unloading overhead. Throughput is significantly higher than the single-GPU version.

import torch, bitsandbytes as bnb
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

print("torch", torch.__version__)
print("bnb", bnb.__version__)

cfg = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4")
m = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-7B-Instruct",
    quantization_config=cfg,
    device_map={"": 0},
    low_cpu_mem_usage=True,
)
print(type(m.model.layers[0].mlp.gate_proj))

In [1]:
# Run this BEFORE importing torch/transformers in a fresh kernel:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

Library	Purpose
Queue	manage tasks between workers
typing	define data types
AutoModelForCausalLM	load language model
AutoTokenizer	convert text ↔ tokens
BitsAndBytesConfig	load model in 4-bit/8-bit
hf_logging	control Hugging Face logs
login	authenticate to Hugging Fac

In [2]:
import torch
import json
import time
import threading
import re
from queue import Queue
from typing import List, Optional, Literal
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from transformers import logging as hf_logging
from huggingface_hub import login

# Set seed for reproducibility
torch.manual_seed(42)


In [3]:
!nvidia-smi

Sun Mar 29 18:39:23 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 560.35.03              Driver Version: 560.35.03      CUDA Version: 12.6     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:01:00.0 Off |                    0 |
| N/A   23C    P0             51W /  400W |       5MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [4]:
def _recover_extractions_from_truncated_json(cleaned: str):
    marker = '"extractions"'
    marker_idx = cleaned.find(marker)
    if marker_idx == -1:
        return None

    list_start = cleaned.find('[', marker_idx)
    if list_start == -1:
        return None

    items = []
    obj_start = None
    depth = 0
    in_string = False
    escape = False

    for idx in range(list_start + 1, len(cleaned)):
        ch = cleaned[idx]
        if in_string:
            if escape:
                escape = False
            elif ch == '\\':
                escape = True
            elif ch == '"':
                in_string = False
            continue

        if ch == '"':
            in_string = True
            continue

        if ch == '{':
            if depth == 0:
                obj_start = idx
            depth += 1
        elif ch == '}':
            if depth > 0:
                depth -= 1
                if depth == 0 and obj_start is not None:
                    snippet = cleaned[obj_start:idx + 1]
                    try:
                        items.append(json.loads(snippet))
                    except Exception:
                        pass
                    obj_start = None

    if items:
        # ADDED: salvage complete extraction objects from truncated Agent A output.
        return {"extractions": items}
    return None


def _safe_json_load(text: str):
    if not text:
        return None
    cleaned = text.strip()
    cleaned = re.sub(r"^```json\s*", "", cleaned, flags=re.IGNORECASE)
    cleaned = re.sub(r"^```\s*", "", cleaned)
    cleaned = re.sub(r"\s*```$", "", cleaned).strip()

    try:
        return json.loads(cleaned)
    except Exception:
        pass

    start = cleaned.find("{")
    end = cleaned.rfind("}")
    if start != -1 and end != -1 and end > start:
        try:
            return json.loads(cleaned[start:end + 1])
        except Exception:
            pass

    repaired = _recover_extractions_from_truncated_json(cleaned)
    if repaired is not None:
        return repaired
    return None

In [5]:
# Enable Hugging Face progress bars inside Jupyter
hf_logging.enable_progress_bar()

Step 1: Accept the License Agreement
Open your web browser and go to: https://huggingface.co/google/gemma-2-27b-it

Create a free Hugging Face account (or log in if you have one).

At the top of the Gemma 2 page, you will see a big box asking you to share your contact info and "Acknowledge License". Click agree.
(Approval is usually instant).

Step 2: Create an Access Token
Your Jupyter Notebook needs a "key" to prove to Hugging Face that you accepted the license.

On Hugging Face, click your profile picture in the top right and go to Settings.

On the left menu, click Access Tokens.

Click Create new token.

Give it a name (like "Server"), set the Type to Read, and generate it.

Copy the long token starting with hf_...

In [6]:
# Paste your token inside the quotes
#login(token="hf_YOUR_COPIED_TOKEN_HERE")
login(token="")

In [7]:
# To check whihc virtual env I am in
import sys
print(sys.executable)

/home/m199589/miniconda3-2/envs/R0/bin/python


In [8]:
print(torch.cuda.get_device_name(0))
print("Allocated:", torch.cuda.memory_allocated(0)/1e9, "GB")
print("Reserved:", torch.cuda.memory_reserved(0)/1e9, "GB")

NVIDIA A100-SXM4-40GB
Allocated: 0.0 GB
Reserved: 0.0 GB


1. Model Loading (Permanent)

Unlike the single-GPU version, we load models permanently to specific devices.

load_permanent_model takes a gpu_id (0 or 1) and forces the model to stay there.

We use 4-bit quantization to ensure 32B/27B models fit comfortably within 24GB VRAM.

In [9]:
import os, gc, torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# must be set before model load
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

def load_permanent_model(model_id: str, local_gpu_id: int):
    device = f"cuda:{local_gpu_id}"
    print(f"[System] Loading {model_id} on {device}...")

    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )

    tok = AutoTokenizer.from_pretrained(model_id, use_fast=True)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        quantization_config=bnb_config,
        device_map="auto",
        max_memory={local_gpu_id: "28GiB", "cpu": "160GiB"},
        low_cpu_mem_usage=True,
        offload_folder=f"/tmp/offload_{model_id.split('/')[-1]}_{local_gpu_id}",
        offload_state_dict=True,
        use_safetensors=True,
        trust_remote_code=True,
        # IMPORTANT: do not pass dtype/torch_dtype here
    )

    model.eval()
    print("Loaded layer type:", type(model.model.layers[0].mlp.gate_proj))
    return model, tok, device

2. Worker Function: Agent A (Extractor on GPU 0)

This function runs in its own thread. It pulls raw abstracts from the input_queue, processes them using the model on GPU 0, and pushes the result to the audit_queue.

In [10]:
def agent_a_worker(input_queue, audit_queue, model, tokenizer, device):
    """
    GPU 0 Worker: Reads JSON entries and extracts epidemiological fields.
    """
    print(f"[Agent A] Ready on {device}. Processing abstracts...")

 
    system_prompt = """
            You are an expert Epidemiologist.
            
            From the abstract, extract ONLY values explicitly referring to:
            - Basic reproduction number (R0)
            - Effective reproduction number (Rt)
            
            Ignore any other epidemiological metrics.

            Extract ONLY concrete reported estimates or explicitly reported ranges/inequalities tied to a study result.
            Do NOT extract conceptual definitions, threshold explanations, generic statements like 'R0 > 1', methodological discussion, or mentions of R0/Rt without a reported estimate.
            If the abstract discusses R0/R/Rt only conceptually and does not report a study-specific estimate, return {}.
            
            An article may report multiple R0 and/or Rt values.
            Extract EACH value separately.
            
            For each extracted value, return:
            
            1. Disease Name (e.g., COVID-19, Ebola, Mpox)
            2. Location (e.g., Italy, Wuhan, Global, specific country/region)
            3. Metric Type ("R0" or "Rt")
            4. Value (exact reported value, range, or approximation as written in the abstract)
            
            5. Time Frame (specific period associated with the estimate, e.g., "March 2020", "early outbreak phase 2019", "between Jan-Feb 2021"; otherwise "")
            
            Time frame extraction rules:
            - Extract the MOST COMPLETE time frame possible, even if parts appear in different sentences.
            - Combine fragmented temporal expressions into a single unified time frame.
            - If a date range is missing a year, infer the year from nearby context.
            - Prefer specific ranges (e.g., "29 May to 14 July 2009") over vague references (e.g., "May 2009").
            - Always include the year if it can be inferred.
            
            6. Context Label (subgroup/setting label like school A/B, age group, method arm; otherwise "")
            
            7. CI (95% confidence interval text/range if reported; otherwise "")
            
            CI extraction rules:
            - Extract ONLY confidence intervals explicitly labeled as CI or 95% CI.
            - DO NOT treat IQR as CI.
            
            8. IQR (interquartile range if reported; otherwise "")
            
            IQR extraction rules:
            - Extract values explicitly labeled as IQR.
            - Do NOT confuse IQR with CI or other intervals.
            
            9. Evidence Snippet (exact quote supporting the value)

            Evidence snippet rules:
            - Keep it SHORT: use the shortest exact quote that still supports the value.
            - Do not copy long multi-clause sentences if a shorter exact span is enough.
            - Keep JSON compact and avoid unnecessary whitespace.
            
            Return ONLY valid JSON in the following format.
            Do not include any explanation, notes, or text before or after the JSON.
            
            {
              "extractions": [
                {
                  "disease": "string",
                  "location": "string",
                  "metric_type": "R0" or "Rt",
                  "value": "string",
                  "time_frame": "string",
                  "context_label": "string",
                  "ci": "string",
                  "iqr": "string",
                  "evidence_snippet": "string"
                }
              ]
            }
            
            If no R0 or Rt values are found, return:
            {}
            """






    
    while True:
        entry = input_queue.get()
        try:
            if entry is None:
                audit_queue.put(None)
                break

            metadata = dict(entry)
            metadata.setdefault("retry_count", 0)
            abstract_text = metadata.get("text", "")
            feedback = metadata.get("audit_feedback")
            user_content = f"Abstract:\n{abstract_text}"
            if feedback:
                user_content += f"\n\nPrevious verifier feedback:\n{feedback}\nPlease correct your extraction."

            messages = [
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_content},
            ]

            text_input = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
            inputs = tokenizer(text_input, return_tensors="pt").to(device)

            with torch.no_grad():
                generated_ids = model.generate(
                    **inputs,
                    max_new_tokens=1024,
                    do_sample=False  # CHANGED: keep deterministic; removed temperature warning case
                )

            output_text = tokenizer.decode(
                generated_ids[0][len(inputs.input_ids[0]):],
                skip_special_tokens=True
            )

            pmid = metadata.get("pmid", "Unknown")
            print(f"\n[Agent A RAW] PMID: {pmid}\n{output_text}\n")

            data = _safe_json_load(output_text)
            print(f"[Agent A PARSED] PMID: {pmid} | parsed_ok={isinstance(data, dict)} | data={data}")

            required = {"disease", "location", "metric_type", "value", "time_frame", "context_label", "ci", "iqr","evidence_snippet"}

            # Accept the list schema and send one payload per abstract to Agent B.
            if isinstance(data, dict) and data == {}:
                audit_queue.put({
                    "metadata": metadata,
                    "extraction": None,
                    "no_evidence": True,
                    "agent_a_raw_output": output_text
                })

            elif isinstance(data, dict) and isinstance(data.get("extractions"), list):
                extractions = data.get("extractions", [])
                valid_items = []
                for item in extractions:
                    if not isinstance(item, dict):
                        continue
                    item.setdefault("time_frame", "")
                    item.setdefault("context_label", "")
                    value = item.get("value")
                    if required.issubset(item.keys()) and value not in (None, ""):
                        valid_items.append(item)

                if not extractions:
                    audit_queue.put({
                        "metadata": metadata,
                        "extraction": None,
                        "no_evidence": True,
                        "agent_a_raw_output": output_text
                    })
                elif not valid_items:
                    empty_value_only = all(
                        isinstance(item, dict) and item.get("value") in (None, "")
                        for item in extractions
                    )
                    if empty_value_only:
                        # CHANGED: treat empty-value pseudo-extractions as no evidence, not as an extraction failure.
                        audit_queue.put({
                            "metadata": metadata,
                            "extraction": None,
                            "no_evidence": True,
                            "agent_a_raw_output": output_text
                        })
                    else:
                        audit_queue.put({
                            "metadata": metadata,
                            "extraction": None,
                            "agent_a_error": "Extraction items present but invalid schema/value",
                            "agent_a_raw_output": output_text
                        })
                else:
                    audit_queue.put({
                        "metadata": metadata,
                        "extractions": valid_items,
                        "agent_a_raw_output": output_text
                    })

            elif isinstance(data, dict):
                data.setdefault("time_frame", "")
                data.setdefault("context_label", "")
                if required.issubset(data.keys()) and data.get("value") not in (None, ""):
                    audit_queue.put({
                        "metadata": metadata,
                        "extractions": [data],
                        "agent_a_raw_output": output_text
                    })
                else:
                    if data.get("value") in (None, ""):
                        audit_queue.put({
                            "metadata": metadata,
                            "extraction": None,
                            "no_evidence": True,
                            "agent_a_raw_output": output_text
                        })
                    else:
                        audit_queue.put({
                            "metadata": metadata,
                            "extraction": None,
                            "agent_a_error": "Invalid or empty extraction JSON from Agent A",
                            "agent_a_raw_output": output_text
                        })

            else:
                audit_queue.put({
                    "metadata": metadata,
                    "extraction": None,
                    "agent_a_error": "Invalid or empty extraction JSON from Agent A",
                    "agent_a_raw_output": output_text
                })

        except Exception as e:
            metadata = dict(entry) if isinstance(entry, dict) else {"pmid": "Unknown", "retry_count": 0}
            audit_queue.put({
                "metadata": metadata,
                "extraction": None,
                "agent_a_error": f"Agent A runtime error: {e}",
                "agent_a_raw_output": ""
            })
        finally:
            input_queue.task_done()

    print("\n[Agent A] Shutting down.")


3. Worker Function: Agent B (Critic on GPU 1)

This function runs in a separate thread. It pulls extracted data from the audit_queue, critiques it using the model on GPU 1, and saves valid results to the final_database.

In [11]:
def agent_b_worker(audit_queue, final_database, human_review_database, retry_review_database, input_queue, model, tokenizer, device):
    print(f"[Agent B] Ready on {device}. Auditing...")

    audit_instruction = (
    "Return exactly one line: PASS or FAIL: <short reason>.\n\n"
    "Check whether Agent A correctly extracted all R0/Rt-related information from the abstract. "
    "Dependencies include disease, location, time_frame, context_label, CI, IQR, and evidence_snippet. "
    "If a dependency is present in the text but missing or incomplete in Agent A's extraction, return FAIL. "
    "If the abstract mentions R0/R/Rt only conceptually, as a definition, threshold rule, or general discussion without a concrete study-specific estimate, PASS with no evidence is correct. "
    "Do NOT penalize if a field is truly not present in the text. "
    "Do NOT treat IQR as CI. If IQR exists but CI does not, this is correct behavior."
    )

    while True:
        payload = audit_queue.get()
        try:
            if payload is None:
                break

            metadata = payload.get("metadata", {})
            agent_a_error = payload.get("agent_a_error")
            no_evidence = payload.get("no_evidence", False)

            extractions = payload.get("extractions")
            if extractions is None:
                # Backward compatibility for old payloads with a single extraction field.
                single = payload.get("extraction")
                if isinstance(single, dict):
                    extractions = [single]
                else:
                    extractions = []

            if agent_a_error:
                status, reason = "FAIL", agent_a_error
                raw_verdict = f"FAIL: {reason}"
            else:
                user_msg = (
                    f"{audit_instruction}\n\n"
                    f"Abstract:\n{metadata.get('text', '')}\n\n"
                    f"Agent A Extractions:\n{json.dumps(extractions, ensure_ascii=False)}\n"
                    f"Agent A no_evidence flag: {no_evidence}\n"
                )
                # CHANGED: use the compact Qwen-friendly verifier prompt that worked in standalone testing.
                messages = [{"role": "user", "content": user_msg}]
                text_input = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
                inputs = tokenizer(text_input, return_tensors="pt").to(device)

                with torch.no_grad():
                    out = model.generate(
                        **inputs,
                        max_new_tokens=64,
                        do_sample=False,
                        pad_token_id=tokenizer.eos_token_id,
                    )

                raw_verdict = tokenizer.decode(out[0][len(inputs.input_ids[0]):], skip_special_tokens=True).strip()
                raw_verdict = (raw_verdict or "").strip()
                print(f"[Agent B RAW] PMID: {metadata.get('pmid')} | verdict={raw_verdict!r}")

                if not raw_verdict:
                    # CHANGED: retry once with a compact prompt before escalating verifier-empty cases.
                    fallback_user_msg = (
                        "Return exactly one line only.\n"
                        "PASS if Agent A's extraction is correct, otherwise FAIL: <short reason>.\n\n"
                        "If the abstract contains only conceptual/threshold discussion of R0/R/Rt and no concrete estimate, PASS is correct when no extraction is present.\n\n"
                        f"Abstract:\n{metadata.get('text', '')}\n\n"
                        f"Extractions:\n{json.dumps(extractions, ensure_ascii=False)}\n"
                        f"No evidence flag: {no_evidence}\n"
                    )
                    fallback_messages = [
                        {"role": "user", "content": fallback_user_msg},
                    ]
                    fallback_text_input = tokenizer.apply_chat_template(fallback_messages, tokenize=False, add_generation_prompt=True)
                    fallback_inputs = tokenizer(fallback_text_input, return_tensors="pt").to(device)

                    with torch.no_grad():
                        fallback_out = model.generate(
                            **fallback_inputs,
                            max_new_tokens=32,
                            do_sample=False,
                            pad_token_id=tokenizer.eos_token_id,
                        )

                    raw_verdict = tokenizer.decode(
                        fallback_out[0][len(fallback_inputs.input_ids[0]):],
                        skip_special_tokens=True
                    ).strip()
                    raw_verdict = (raw_verdict or "").strip()
                    print(f"[Agent B RAW RETRY] PMID: {metadata.get('pmid')} | verdict={raw_verdict!r}")

                    if not raw_verdict:
                        status, reason = "FAIL", "Verifier empty output"
                        raw_verdict = "FAIL: Verifier empty output"
                    else:
                        status, reason = parse_pass_fail(raw_verdict)
                else:
                    status, reason = parse_pass_fail(raw_verdict)

            if status == "PASS":
                if len(extractions) > 0:
                    for ext in extractions:
                        fixed = normalize_extraction(ext)
                        final_record = {
                            "article_id": metadata.get("pmid"),
                            "year": metadata.get("year"),
                            "disease": fixed.get("disease"),
                            "location": fixed.get("location"),
                            "metric": fixed.get("metric_type"),
                            "value": fixed.get("value"),
                            "time_frame": fixed.get("time_frame"),
                            "context_label": fixed.get("context_label"),
                            "ci": fixed.get("ci"),
                            # ADDED: preserve the extracted IQR in validated output.
                            "iqr": fixed.get("iqr"),
                            "evidence_snippet": fixed.get("evidence_snippet"),
                            "agent_a_extraction": ext,
                            "agent_b_verdict": raw_verdict,
                        }
                        final_database.append(final_record)
                    print(f"[PASS] PMID: {metadata.get('pmid')} | extracted={len(extractions)}")
                else:
                    final_record = {
                        "article_id": metadata.get("pmid"),
                        "year": metadata.get("year"),
                        "disease": None,
                        "location": None,
                        "metric": None,
                        "value": None,
                        "time_frame": None,
                        "context_label": None,
                        "ci": None,
                        "iqr": None,
                        "evidence_snippet": None,
                        "agent_a_extraction": None,
                        "agent_b_verdict": raw_verdict,
                        "no_evidence": True,
                    }
                    final_database.append(final_record)
                    print(f"[PASS-NO-EVIDENCE] PMID: {final_record['article_id']}")
            else:
                retry_count = int(metadata.get("retry_count", 0))
                # CHANGED: send to human review when retry_count reaches 2.
                if retry_count >= 2:
                    fail_item = dict(metadata)
                    fail_item["failure_reason"] = reason
                    fail_item["agent_a_extractions"] = extractions
                    fail_item["agent_b_verdict"] = raw_verdict
                    fail_item["no_evidence"] = no_evidence
                    human_review_database.append(fail_item)
                    print(f"[FAIL] PMID: {metadata.get('pmid')} | reason={reason}")
                else:
                    metadata["retry_count"] = retry_count + 1
                    # CHANGED: include the verifier reason so Agent A can correct the specific problem.
                    metadata["audit_feedback"] = f"Agent B flagged this abstract-level extraction: {reason}"
                    # ADDED: keep a retry trail for debugging borderline abstracts.
                    retry_review_database.append({
                        "pmid": metadata.get("pmid"),
                        "year": metadata.get("year"),
                        "retry_count_after_decision": metadata.get("retry_count"),
                        "failure_reason": reason,
                        "agent_b_verdict": raw_verdict,
                        "agent_a_extractions": extractions,
                        "no_evidence": no_evidence,
                    })
                    input_queue.put(metadata)
                    print(f"[RETRY] PMID: {metadata.get('pmid')} | reason={reason}")
        finally:
            audit_queue.task_done()

    print("\n[Agent B] Shutting down.")


4. Main Execution Block

This block initializes the entire system.

Loads Agent A onto GPU 0.

Loads Agent B onto GPU 1.

Creates the communication queues.

Starts the worker threads.

Feeds the data and waits for completion.

In [12]:
!nvidia-smi

Sun Mar 29 18:39:56 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 560.35.03              Driver Version: 560.35.03      CUDA Version: 12.6     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:01:00.0 Off |                    0 |
| N/A   24C    P0             50W /  400W |       5MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [13]:
# --- 1. Load Models (This takes time, do it once) ---
# GPU 0: Extractor (Qwen)
model_a, tok_a, dev_a = load_permanent_model("Qwen/Qwen2.5-32B-Instruct", 0)
 

[System] Loading Qwen/Qwen2.5-32B-Instruct on cuda:0...


Loading checkpoint shards:   0%|          | 0/17 [00:00<?, ?it/s]

Loaded layer type: <class 'bitsandbytes.nn.modules.Linear4bit'>


In [14]:
# GPU 1: Critic (Qwen)
model_b, tok_b, dev_b = load_permanent_model("Qwen/Qwen2.5-7B-Instruct", 1)


[System] Loading Qwen/Qwen2.5-7B-Instruct on cuda:1...


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Loaded layer type: <class 'bitsandbytes.nn.modules.Linear4bit'>


In [43]:
#Load Data from JSON
try:
        with open("../data/raw/raw_abstracts.json", "r") as f:
            raw_data = json.load(f)
        print(f"\n[System] Successfully loaded {len(raw_data)} abstracts from raw_abstracts.json")
except FileNotFoundError:
        print("\n[Error] Could not find 'raw_abstracts.json'. Please ensure the file is in the same folder.")
        raw_data = []



[System] Successfully loaded 246 abstracts from raw_abstracts.json


In [44]:
if raw_data:
    raw_data.sort(key=lambda x: x.get("pmid", ""))
    raw_data = raw_data[40:60]

    print(f"[System] Kept first {len(raw_data)} documents after sorting.")

[System] Kept first 20 documents after sorting.


In [45]:
print

<function print>

In [46]:
#raw_data=raw_data[:10]

In [47]:
print(len(raw_data))

20


In [48]:
print(raw_data[0])

{'disease_query': 'Ebola', 'pmid': '27806049', 'title': 'Containing Ebola at the Source with Ring Vaccination.', 'year': '2016', 'text': 'Title: Containing Ebola at the Source with Ring Vaccination.\nAbstract: Interim results from the Guinea Ebola ring vaccination trial suggest high efficacy of the rVSV-ZEBOV vaccine. These findings open the door to the use of ring vaccination strategies in which the contacts and contacts of contacts of each index case are promptly vaccinated to contain future Ebola virus disease outbreaks. To provide a numerical estimate of the effectiveness of ring vaccination strategies we introduce a spatially explicit agent-based model to simulate Ebola outbreaks in the Pujehun district, Sierra Leone, structurally similar to previous modelling approaches. We find that ring vaccination can successfully contain an outbreak for values of the effective reproduction number up to 1.6. Through an extensive sensitivity analysis of parameters characterising the readiness a

In [49]:
def parse_pass_fail(text: str):
    t = (text or "").strip()
    first = t.splitlines()[0].strip().upper() if t else ""
    if "PASS" in first:
        return "PASS", "Accepted by verifier"
    if "FAIL" in first:
        reason = t.split(":", 1)[1].strip() if ":" in t else "Verifier rejected extraction"
        return "FAIL", reason
    if "PASS" in t.upper():
        return "PASS", "Accepted by verifier"
    if "FAIL" in t.upper():
        return "FAIL", "Verifier rejected extraction"
    return "FAIL", f"Verifier unclear: {t[:120]}"

In [50]:
def normalize_extraction(extraction: dict):
    if not isinstance(extraction, dict):
        return None
    metric = str(extraction.get("metric_type", "")).strip().upper()
    metric = "R0" if metric == "R0" else ("Rt" if metric == "RT" else extraction.get("metric_type"))
    try:
        value = float(extraction.get("value"))
    except Exception:
        value = extraction.get("value")
    return {
        "disease": extraction.get("disease"),
        "location": extraction.get("location"),
        "metric_type": metric,
        "value": value,
        "time_frame": extraction.get("time_frame"),
        "context_label": extraction.get("context_label"),
        "ci": extraction.get("ci"),
        # ADDED: keep IQR alongside CI instead of dropping it during normalization.
        "iqr": extraction.get("iqr"),
        "evidence_snippet": extraction.get("evidence_snippet"),
    }


In [51]:
# 3. Setup Queues
q_in = Queue()
q_audit = Queue()
results_db = []
# --- CHANGED: List to store abstracts that reach retry_count 2
human_review_db = [] 
# --- ADDED: List to store every retry decision for later debugging.
retry_review_db = []

# 4. Start Threads
t1 = threading.Thread(target=agent_a_worker, args=(q_in, q_audit, model_a, tok_a, dev_a))
# --- CHANGED: Passed human_review_db, retry_review_db, and q_in as arguments to Agent B
t2 = threading.Thread(target=agent_b_worker, args=(q_audit, results_db, human_review_db, retry_review_db, q_in, model_b, tok_b, dev_b))

t1.start()
t2.start()

# 5. Feed the Dictionaries into the Queue
print(f"\n[System] Feeding data to pipelines...")
for entry in raw_data:
    q_in.put(entry) 

[Agent A] Ready on cuda:0. Processing abstracts...
[Agent B] Ready on cuda:1. Auditing...

[System] Feeding data to pipelines...


/home/m199589/miniconda3-2/envs/R0/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:628: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/home/m199589/miniconda3-2/envs/R0/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:633: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/home/m199589/miniconda3-2/envs/R0/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:650: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(



[Agent A RAW] PMID: 27806049
{}

[Agent A PARSED] PMID: 27806049 | parsed_ok=True | data={}

[Agent A RAW] PMID: 27840011
{}

[Agent A PARSED] PMID: 27840011 | parsed_ok=True | data={}
[Agent B RAW] PMID: 27806049 | verdict='PASS: Abstract mentions R0/R/Rt only conceptually without a concrete study-specific estimate.'
[PASS-NO-EVIDENCE] PMID: 27806049

[Agent A RAW] PMID: 27846234
{}

[Agent A PARSED] PMID: 27846234 | parsed_ok=True | data={}

[Agent A RAW] PMID: 28202592
{}

[Agent A PARSED] PMID: 28202592 | parsed_ok=True | data={}
[Agent B RAW] PMID: 27840011 | verdict='PASS: Abstract does not contain any R0/Rt-related information.'
[PASS-NO-EVIDENCE] PMID: 27840011

[Agent A RAW] PMID: 28252663
{}

[Agent A PARSED] PMID: 28252663 | parsed_ok=True | data={}
[Agent B RAW] PMID: 27846234 | verdict='PASS: No R0/Rt-related information was mentioned in the abstract.'
[PASS-NO-EVIDENCE] PMID: 27846234
[Agent B RAW] PMID: 28202592 | verdict='PASS: No R0/Rt-related information is mentioned

In [54]:
# --- CHANGED: Wait for the queues to fully empty (including retries) BEFORE sending the stop signal.
q_in.join() 
q_audit.join()

# --- CHANGED: Now that processing is completely done, safely send the stop signal.
q_in.put(None)

# 6. Wait & Save
t1.join()
t2.join()


from datetime import datetime

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
validated_path = f"../data/processed/validated_results_{timestamp}.json"
human_review_path = f"../data/processed/human_review_required_{timestamp}.json"
retry_review_path = f"../data/processed/retry_review_{timestamp}.json"

with open(validated_path, "w") as f:
    json.dump(results_db, f, indent=2)

# --- ADDED: Save the human review list to a separate file
with open(human_review_path, "w") as f:
    json.dump(human_review_db, f, indent=2)

# --- ADDED: Save the retry trail for debugging repeated verifier failures.
with open(retry_review_path, "w") as f:
    json.dump(retry_review_db, f, indent=2)

print(
    f"\n[System] Done! Saved {len(results_db)} valid records to {validated_path}, "
    f"{len(human_review_db)} records for human review to {human_review_path}, "
    f"and {len(retry_review_db)} retry events to {retry_review_path}."
)

 


[System] Done! Saved 30 valid records to ../data/processed/validated_results_20260329_200954.json, 0 records for human review to ../data/processed/human_review_required_20260329_200954.json, and 1 retry events to ../data/processed/retry_review_20260329_200954.json.


In [55]:
pass_with_value = sum(1 for r in results_db if r.get("metric") is not None)
pass_no_evidence = sum(1 for r in results_db if r.get("metric") is None)

print(
    f"\n[System] Done! Saved {len(results_db)} accepted records "
    f"({pass_with_value} PASS, {pass_no_evidence} PASS-NO-EVIDENCE) "
    f"and {len(human_review_db)} records for human review."
)



[System] Done! Saved 30 accepted records (15 PASS, 15 PASS-NO-EVIDENCE) and 0 records for human review.


In [56]:
# --- ADDED: Export validated results to Excel sorted by abstract title and article ID.
import pandas as pd

title_lookup = {str(item.get("pmid")): item.get("title") for item in raw_data}
excel_rows = []
for row in results_db:
    item = dict(row)
    item["title"] = title_lookup.get(str(item.get("article_id")), "")
    excel_rows.append(item)

excel_df = pd.DataFrame(excel_rows)
column_order = [
    "title",
    "article_id",
    "year",
    "disease",
    "location",
    "metric",
    "value",
    "time_frame",
    "context_label",
    "ci",
    "iqr",
    "evidence_snippet",
    "agent_b_verdict",
    "no_evidence",
    "agent_a_extraction",
]
remaining_cols = [c for c in excel_df.columns if c not in column_order]
excel_df = excel_df[[c for c in column_order if c in excel_df.columns] + remaining_cols]
excel_df = excel_df.sort_values(by=["title", "article_id"], na_position="last")

excel_path = f"../data/processed/validated_results_{timestamp}.xlsx"
excel_df.to_excel(excel_path, index=False)
print(f"[System] Excel export saved to {excel_path} with {len(excel_df)} rows.")


[System] Excel export saved to ../data/processed/validated_results_20260329_200954.xlsx with 30 rows.
